# Notebook 05 — Feature Engineering & Similarity Representation

## Patient Similarity Network + Agent-Based Modelling for Readmission

### Purpose

Convert the patient-level representation into a clinically interpretable numerical feature
space and validate alternative patient-similarity definitions before constructing the network.

This notebook follows the project proposal:

- encode categorical variables;
- scale numerical variables;
- create a clinically interpretable patient feature vector;
- test cosine similarity;
- test Euclidean distance after scaling;
- optionally test a mixed-type similarity;
- inspect nearest neighbours for randomly selected patients.

**No patient network is constructed here.** Notebook 06 will construct the sparse weighted
patient similarity network only after this representation passes validation.

### Critical leakage rule

The readmission outcome is never used to build the feature vectors or similarity scores.

Preprocessing parameters are fitted on the **training patients only** and then applied to
validation/test patients.

## Expected project structure

```text
sna/
├── diabetes+130-us+hospitals+for+years+1999-2008/
├── notebooks/
│   ├── 01_dataset_audit.ipynb
│   ├── 02_cleaning.ipynb
│   ├── 03_patient_representation.ipynb
│   ├── 04_split_leakage_check.ipynb
│   └── 05_features_similarity.ipynb
├── results/
└── figures/
```

### Inputs

- `results/03_patient_representation.csv`
- `results/04_train_patient_ids.csv`
- `results/04_validation_patient_ids.csv`
- `results/04_test_patient_ids.csv`

### Outputs

- transformed train / validation / test feature matrices;
- feature dictionary;
- preprocessing metadata;
- similarity comparison table;
- nearest-neighbour validation table;
- validation summary.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances

SEED = 42
N_NEIGHBORS = 6          # 1 query patient + 5 neighbours
N_VALIDATION_PATIENTS = 5

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd()
candidate_roots = [cwd, cwd.parent, cwd.parent.parent]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / "results" / "03_patient_representation.csv").exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find results/03_patient_representation.csv. "
        "Open this notebook inside the sna project folder."
    )

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PATIENT_PATH = RESULTS_DIR / "03_patient_representation.csv"
TRAIN_IDS_PATH = RESULTS_DIR / "04_train_patient_ids.csv"
VALID_IDS_PATH = RESULTS_DIR / "04_validation_patient_ids.csv"
TEST_IDS_PATH = RESULTS_DIR / "04_test_patient_ids.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: c:\Users\Gayatri\OneDrive\Desktop\sna


In [2]:
patient_df = pd.read_csv(PATIENT_PATH, low_memory=False)

train_ids = pd.read_csv(TRAIN_IDS_PATH)["patient_nbr"].tolist()
valid_ids = pd.read_csv(VALID_IDS_PATH)["patient_nbr"].tolist()
test_ids = pd.read_csv(TEST_IDS_PATH)["patient_nbr"].tolist()

train_ids = set(train_ids)
valid_ids = set(valid_ids)
test_ids = set(test_ids)

assert patient_df["patient_nbr"].is_unique
assert train_ids.isdisjoint(valid_ids)
assert train_ids.isdisjoint(test_ids)
assert valid_ids.isdisjoint(test_ids)

train_df = patient_df[patient_df["patient_nbr"].isin(train_ids)].copy()
valid_df = patient_df[patient_df["patient_nbr"].isin(valid_ids)].copy()
test_df = patient_df[patient_df["patient_nbr"].isin(test_ids)].copy()

print("All patients:", len(patient_df))
print("Train:", len(train_df))
print("Validation:", len(valid_df))
print("Test:", len(test_df))

All patients: 71518
Train: 50062
Validation: 10728
Test: 10728


# 1. Freeze feature eligibility

The following are never features:

- `patient_nbr` — identifier
- any readmission/outcome variable
- any encounter-level outcome summary

We also exclude variables that Notebook 02/03 explicitly marked as unsuitable for the current
representation, including review-required or high-missingness variables.

The feature matrix is therefore constructed only from the patient-level representation.

In [3]:
OUTCOME_COLS = {
    "readmitted",
    "encounters_no_readmission",
    "encounters_readmission_over_30d",
    "encounters_readmission_under_30d",
    "any_observed_readmission_under_30d",
    "any_observed_readmission",
}

IDENTIFIER_COLS = {"patient_nbr"}

# Columns from Notebook 03 that are not included in the final similarity representation.
REVIEW_OR_HIGH_MISSING_COLS = {
    "discharge_disposition_id",
    "time_in_hospital",
    "weight",
}

all_excluded = OUTCOME_COLS | IDENTIFIER_COLS | REVIEW_OR_HIGH_MISSING_COLS

candidate_feature_cols = [
    c for c in patient_df.columns
    if c not in all_excluded
]

leakage_found = sorted(set(candidate_feature_cols) & OUTCOME_COLS)

print("Candidate feature count:", len(candidate_feature_cols))
print("Excluded outcome/identifier columns:", sorted(OUTCOME_COLS | IDENTIFIER_COLS))
print("Leakage columns:", leakage_found)

assert leakage_found == []
assert "patient_nbr" not in candidate_feature_cols

Candidate feature count: 38
Excluded outcome/identifier columns: ['any_observed_readmission', 'any_observed_readmission_under_30d', 'encounters_no_readmission', 'encounters_readmission_over_30d', 'encounters_readmission_under_30d', 'patient_nbr', 'readmitted']
Leakage columns: []


# 2. Identify numeric and categorical features

Numeric features will be median-imputed and standardized.

Categorical features will be most-frequent imputed and one-hot encoded.

The transformation is fitted **only on the training set**.

This prevents validation/test information from determining imputation values, scaling
parameters, or category vocabularies.

In [4]:
numeric_cols = patient_df[candidate_feature_cols].select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_cols = [
    c for c in candidate_feature_cols
    if c not in numeric_cols
]

print("Numeric features:", len(numeric_cols))
print(numeric_cols)

print("\nCategorical features:", len(categorical_cols))
print(categorical_cols)

assert len(numeric_cols) + len(categorical_cols) == len(candidate_feature_cols)

Numeric features: 32
['encounter_count', 'number_inpatient_sum', 'number_inpatient_mean', 'number_inpatient_max', 'number_emergency_sum', 'number_emergency_mean', 'number_emergency_max', 'number_outpatient_sum', 'number_outpatient_mean', 'number_outpatient_max', 'num_lab_procedures_sum', 'num_lab_procedures_mean', 'num_lab_procedures_max', 'num_procedures_sum', 'num_procedures_mean', 'num_procedures_max', 'num_medications_sum', 'num_medications_mean', 'num_medications_max', 'number_diagnoses_sum', 'number_diagnoses_mean', 'number_diagnoses_max', 'medication_active_entries_sum', 'medication_active_entries_mean', 'medication_active_entries_max', 'distinct_medications_recorded_active', 'diagnosis_entries_sum', 'diagnosis_entries_mean', 'diagnosis_entries_max', 'unique_diagnosis_codes', 'admission_type_id_mode', 'admission_source_id_mode']

Categorical features: 6
['race_mode', 'gender_mode', 'age_mode', 'A1Cresult_mode', 'change_mode', 'diabetesMed_mode']


# 3. Build the leakage-safe preprocessing pipeline

### Numerical branch

`median imputation → standard scaling`

### Categorical branch

`most-frequent imputation → one-hot encoding`

`handle_unknown="ignore"` ensures a category occurring only in validation/test does not
break transformation.

The transformed matrix is kept sparse where possible so that later nearest-neighbour
computations remain feasible.

In [5]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

X_train = preprocessor.fit_transform(train_df[candidate_feature_cols])
X_valid = preprocessor.transform(valid_df[candidate_feature_cols])
X_test = preprocessor.transform(test_df[candidate_feature_cols])

print("Train feature matrix:", X_train.shape)
print("Validation feature matrix:", X_valid.shape)
print("Test feature matrix:", X_test.shape)
print("Train matrix type:", type(X_train).__name__)

Train feature matrix: (50062, 57)
Validation feature matrix: (10728, 57)
Test feature matrix: (10728, 57)
Train matrix type: ndarray


# 4. Validate preprocessing

The transformed train/validation/test matrices must:

- have identical feature dimensions;
- contain only finite values;
- contain no target/identifier columns;
- preserve patient counts;
- be reproducible under the fixed preprocessing pipeline.

In [6]:
def matrix_is_finite(X):
    if sparse.issparse(X):
        return np.isfinite(X.data).all()
    return np.isfinite(X).all()

preprocessing_checks = {
    "train_rows_match": X_train.shape[0] == len(train_df),
    "validation_rows_match": X_valid.shape[0] == len(valid_df),
    "test_rows_match": X_test.shape[0] == len(test_df),
    "same_feature_dimension": (
        X_train.shape[1] == X_valid.shape[1] == X_test.shape[1]
    ),
    "train_finite": matrix_is_finite(X_train),
    "validation_finite": matrix_is_finite(X_valid),
    "test_finite": matrix_is_finite(X_test),
    "no_target_feature_column": not any(
        c in candidate_feature_cols for c in OUTCOME_COLS
    ),
    "patient_id_not_feature": "patient_nbr" not in candidate_feature_cols,
}

for name, passed in preprocessing_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

assert all(preprocessing_checks.values())

PASS | train_rows_match
PASS | validation_rows_match
PASS | test_rows_match
PASS | same_feature_dimension
PASS | train_finite
PASS | validation_finite
PASS | test_finite
PASS | no_target_feature_column
PASS | patient_id_not_feature


# 5. Export the transformed feature matrices

The matrices are saved as sparse `.npz` files.

A separate row-ID file preserves the exact patient order corresponding to each matrix.

This is important because later network construction must know which feature-vector row
belongs to which patient.

In [7]:
TRAIN_MATRIX_PATH = RESULTS_DIR / "05_X_train.npz"
VALID_MATRIX_PATH = RESULTS_DIR / "05_X_validation.npz"
TEST_MATRIX_PATH = RESULTS_DIR / "05_X_test.npz"

TRAIN_ORDER_PATH = RESULTS_DIR / "05_train_feature_patient_ids.csv"
VALID_ORDER_PATH = RESULTS_DIR / "05_validation_feature_patient_ids.csv"
TEST_ORDER_PATH = RESULTS_DIR / "05_test_feature_patient_ids.csv"

# Convert dense matrices to CSR if necessary.
X_train_sparse = sparse.csr_matrix(X_train)
X_valid_sparse = sparse.csr_matrix(X_valid)
X_test_sparse = sparse.csr_matrix(X_test)

sparse.save_npz(TRAIN_MATRIX_PATH, X_train_sparse)
sparse.save_npz(VALID_MATRIX_PATH, X_valid_sparse)
sparse.save_npz(TEST_MATRIX_PATH, X_test_sparse)

pd.DataFrame({
    "patient_nbr": train_df["patient_nbr"].tolist()
}).to_csv(TRAIN_ORDER_PATH, index=False)

pd.DataFrame({
    "patient_nbr": valid_df["patient_nbr"].tolist()
}).to_csv(VALID_ORDER_PATH, index=False)

pd.DataFrame({
    "patient_nbr": test_df["patient_nbr"].tolist()
}).to_csv(TEST_ORDER_PATH, index=False)

print("Feature matrices and row-ID files saved.")

Feature matrices and row-ID files saved.


# 6. Feature dictionary

Create an explicit record of how each source patient-level variable is treated.

This makes the similarity representation interpretable and reproducible.

In [8]:
feature_dictionary_rows = []

for c in numeric_cols:
    feature_dictionary_rows.append({
        "source_variable": c,
        "feature_type": "numeric",
        "transformation": "median imputation + StandardScaler",
        "used_for_similarity": True,
        "fit_on": "train only",
    })

for c in categorical_cols:
    feature_dictionary_rows.append({
        "source_variable": c,
        "feature_type": "categorical",
        "transformation": "most-frequent imputation + one-hot encoding",
        "used_for_similarity": True,
        "fit_on": "train only",
    })

for c in sorted(all_excluded):
    if c in patient_df.columns:
        reason = (
            "identifier" if c in IDENTIFIER_COLS else
            "outcome" if c in OUTCOME_COLS else
            "review/high-missingness exclusion"
        )
        feature_dictionary_rows.append({
            "source_variable": c,
            "feature_type": "excluded",
            "transformation": "none",
            "used_for_similarity": False,
            "fit_on": "not applicable",
            "exclusion_reason": reason,
        })

feature_dictionary = pd.DataFrame(feature_dictionary_rows)

DICT_PATH = RESULTS_DIR / "05_feature_dictionary.csv"
feature_dictionary.to_csv(DICT_PATH, index=False)

display(feature_dictionary.head(20))

,source_variable,feature_type,transformation,used_for_similarity,fit_on,exclusion_reason
0,encounter_count,numeric,median imputation + StandardScaler,True,train only,NaN
1,number_inpatient_sum,numeric,median imputation + StandardScaler,True,train only,NaN
2,number_inpatient_mean,numeric,median imputation + StandardScaler,True,train only,NaN
3,number_inpatient_max,numeric,median imputation + StandardScaler,True,train only,NaN
4,number_emergency_sum,numeric,median imputation + StandardScaler,True,train only,NaN
5,number_emergency_mean,numeric,median imputation + StandardScaler,True,train only,NaN
6,number_emergency_max,numeric,median imputation + StandardScaler,True,train only,NaN
7,number_outpatient_sum,numeric,median imputation + StandardScaler,True,train only,NaN
8,number_outpatient_mean,numeric,median imputation + StandardScaler,True,train only,NaN
9,number_outpatient_max,numeric,median imputation + StandardScaler,True,train only,NaN


# 7. Alternative similarity definitions

We test the two alternatives required by the proposal.

## A. Cosine similarity

For vectors `x` and `y`:

`cosine_similarity = (x · y) / (||x|| ||y||)`

Higher = more similar.

## B. Euclidean distance after preprocessing

`euclidean_distance = sqrt(sum((x_i-y_i)^2))`

Lower = more similar.

The same leakage-safe transformed feature space is used for both.

We do **not** build a dense all-pairs similarity matrix. Nearest-neighbour search is used instead.

In [9]:
# Cosine nearest neighbours
cosine_nn = NearestNeighbors(
    n_neighbors=N_NEIGHBORS,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)
cosine_nn.fit(X_train_sparse)

cosine_distances, cosine_indices = cosine_nn.kneighbors(X_train_sparse)

# Euclidean nearest neighbours
euclidean_nn = NearestNeighbors(
    n_neighbors=N_NEIGHBORS,
    metric="euclidean",
    algorithm="brute",
    n_jobs=-1
)
euclidean_nn.fit(X_train_sparse)

euclidean_distances, euclidean_indices = euclidean_nn.kneighbors(X_train_sparse)

print("Cosine neighbour search completed.")
print("Euclidean neighbour search completed.")

Cosine neighbour search completed.
Euclidean neighbour search completed.


# 8. Compare neighbour stability

For each training patient, compare the five nearest non-self neighbours under cosine
and Euclidean distance.

The Jaccard overlap between the two neighbour sets gives a simple measure of local
stability between the similarity definitions.

This is a diagnostic, not a claim that one metric is universally superior.

In [11]:
# ============================================================
# 8. Compare neighbour stability: Cosine vs Euclidean
# ============================================================

# Make sure neighbour-index outputs are NumPy arrays
cosine_indices = np.asarray(cosine_indices)
euclidean_indices = np.asarray(euclidean_indices)

print("Cosine indices shape:", cosine_indices.shape)
print("Euclidean indices shape:", euclidean_indices.shape)

# We expect:
# (number_of_training_patients, N_NEIGHBORS)
assert cosine_indices.ndim == 2, (
    f"Unexpected cosine_indices shape: {cosine_indices.shape}. "
    "Expected a 2-D array."
)

assert euclidean_indices.ndim == 2, (
    f"Unexpected euclidean_indices shape: {euclidean_indices.shape}. "
    "Expected a 2-D array."
)

assert cosine_indices.shape[0] == len(train_df)
assert euclidean_indices.shape[0] == len(train_df)

overlaps = []

for i in range(len(train_df)):

    # Convert each row explicitly to a 1-D array
    cosine_row = np.asarray(cosine_indices[i]).ravel()
    euclidean_row = np.asarray(euclidean_indices[i]).ravel()

    # Remove the patient itself
    cosine_set = {
        int(x) for x in cosine_row
        if int(x) != i
    }

    euclidean_set = {
        int(x) for x in euclidean_row
        if int(x) != i
    }

    union = cosine_set | euclidean_set
    intersection = cosine_set & euclidean_set

    jaccard = (
        len(intersection) / len(union)
        if union
        else 1.0
    )

    overlaps.append(jaccard)

similarity_stability = pd.DataFrame({
    "metric": ["cosine_vs_euclidean"],
    "mean_neighbor_jaccard": [float(np.mean(overlaps))],
    "median_neighbor_jaccard": [float(np.median(overlaps))],
    "p25_neighbor_jaccard": [float(np.quantile(overlaps, 0.25))],
    "p75_neighbor_jaccard": [float(np.quantile(overlaps, 0.75))]
})

display(similarity_stability)

Cosine indices shape: (50062, 6)
Euclidean indices shape: (50062, 6)


,metric,mean_neighbor_jaccard,median_neighbor_jaccard,p25_neighbor_jaccard,p75_neighbor_jaccard
0,cosine_vs_euclidean,0.700699,0.666667,0.428571,1.0


# 9. Missingness-artifact audit

A major proposal checkpoint is to reject a similarity representation if neighbours are
dominated by artifacts such as missingness.

We therefore measure the original missingness pattern in the candidate feature variables
**before imputation**.

For each patient pair, we calculate missingness-pattern overlap using the Jaccard index:

`intersection(missing sets) / union(missing sets)`

This is only a diagnostic. Missingness is not used as the similarity definition.

In [12]:
raw_feature_missing = train_df[candidate_feature_cols].isna().to_numpy(dtype=bool)

def missingness_jaccard(i, j):
    a = set(np.flatnonzero(raw_feature_missing[i]))
    b = set(np.flatnonzero(raw_feature_missing[j]))
    union = a | b
    if not union:
        return 1.0
    return len(a & b) / len(union)

missingness_rows = []

for i in range(len(train_df)):
    neigh = [int(x) for x in cosine_indices[i] if int(x) != i][:5]
    for rank, j in enumerate(neigh, start=1):
        missingness_rows.append({
            "query_row": i,
            "neighbor_row": j,
            "rank": rank,
            "missingness_jaccard": missingness_jaccard(i, j),
        })

missingness_audit = pd.DataFrame(missingness_rows)

print("Mean missingness-pattern Jaccard among cosine neighbours:",
      round(float(missingness_audit["missingness_jaccard"].mean()), 4))
print("Median:",
      round(float(missingness_audit["missingness_jaccard"].median()), 4))

Mean missingness-pattern Jaccard among cosine neighbours: 0.7668
Median: 1.0


### Interpretation rule for missingness audit

There is no universal numerical cutoff that proves "missingness domination".

Therefore we use this as a **manual diagnostic**:

- very high missingness-pattern similarity together with clinically implausible neighbours → STOP;
- moderate/low missingness overlap with clinically coherent neighbours → continue;
- if missingness is clearly driving neighbourhoods, redesign the representation before Notebook 06.

Do not select a similarity metric using the readmission target.

# 10. Randomly selected nearest-neighbour inspection

The proposal requires inspecting nearest neighbours for randomly selected patients.

We use a fixed seed so the review is reproducible.

Only **training patients** are sampled for this validation. Their readmission outcomes are
not displayed or used to select neighbours.

In [13]:
rng = np.random.default_rng(SEED)

sample_size = min(N_VALIDATION_PATIENTS, len(train_df))
sample_rows = rng.choice(len(train_df), size=sample_size, replace=False)

print("Selected training rows:", sample_rows.tolist())
print("Selected patient IDs:",
      train_df.iloc[sample_rows]["patient_nbr"].tolist())

Selected training rows: [38743, 21970, 32767, 4467, 21677]
Selected patient IDs: [90029403, 41787243, 74048202, 3739725, 41492097]


In [14]:
# Compact clinical/context columns for human inspection.
inspection_cols = [
    c for c in [
        "patient_nbr",
        "encounter_count",
        "number_inpatient_sum",
        "number_emergency_sum",
        "number_outpatient_sum",
        "num_medications_mean",
        "number_diagnoses_mean",
        "unique_diagnosis_codes",
        "distinct_medications_recorded_active",
        "race_mode",
        "gender_mode",
        "age_mode",
        "admission_type_id_mode",
        "admission_source_id_mode",
        "A1Cresult_mode",
        "diabetesMed_mode",
    ]
    if c in train_df.columns
]

display(train_df.iloc[sample_rows][inspection_cols])

,patient_nbr,encounter_count,number_inpatient_sum,number_emergency_sum,number_outpatient_sum,num_medications_mean,number_diagnoses_mean,unique_diagnosis_codes,distinct_medications_recorded_active,race_mode,gender_mode,age_mode,admission_type_id_mode,admission_source_id_mode,A1Cresult_mode,diabetesMed_mode
55426,90029403,1,0,0,0,10.0,9.0,3.0,1,Caucasian,Male,[70-80),2,7,>7,Yes
31530,41787243,1,0,0,0,12.0,8.0,3.0,2,Caucasian,Female,[20-30),3,1,Norm,Yes
46889,74048202,1,0,0,2,10.0,9.0,3.0,1,AfricanAmerican,Male,[70-80),1,7,NaN,Yes
6398,3739725,2,1,0,0,32.0,9.0,6.0,1,Caucasian,Female,[60-70),1,7,>7,No
31102,41492097,1,0,0,0,19.0,9.0,3.0,2,Caucasian,Female,[80-90),1,7,NaN,Yes


In [15]:
neighbor_review_rows = []

for query_row in sample_rows:
    cosine_neigh = [
        int(x) for x in cosine_indices[query_row]
        if int(x) != int(query_row)
    ][:5]

    euclidean_neigh = [
        int(x) for x in euclidean_indices[query_row]
        if int(x) != int(query_row)
    ][:5]

    for rank, row in enumerate(cosine_neigh, start=1):
        neighbor_review_rows.append({
            "query_patient_nbr": int(train_df.iloc[query_row]["patient_nbr"]),
            "metric": "cosine",
            "rank": rank,
            "neighbor_patient_nbr": int(train_df.iloc[row]["patient_nbr"]),
            "distance": float(cosine_distances[query_row][
                list(cosine_indices[query_row]).index(row)
            ]),
            "similarity": float(1 - cosine_distances[query_row][
                list(cosine_indices[query_row]).index(row)
            ]),
            "missingness_jaccard": missingness_jaccard(query_row, row),
        })

    for rank, row in enumerate(euclidean_neigh, start=1):
        neighbor_review_rows.append({
            "query_patient_nbr": int(train_df.iloc[query_row]["patient_nbr"]),
            "metric": "euclidean",
            "rank": rank,
            "neighbor_patient_nbr": int(train_df.iloc[row]["patient_nbr"]),
            "distance": float(euclidean_distances[query_row][
                list(euclidean_indices[query_row]).index(row)
            ]),
            "similarity": np.nan,
            "missingness_jaccard": missingness_jaccard(query_row, row),
        })

neighbor_review = pd.DataFrame(neighbor_review_rows)

NEIGHBOR_REVIEW_PATH = RESULTS_DIR / "05_nearest_neighbor_review.csv"
neighbor_review.to_csv(NEIGHBOR_REVIEW_PATH, index=False)

display(neighbor_review)

,query_patient_nbr,metric,rank,neighbor_patient_nbr,distance,similarity,missingness_jaccard
0,90029403,cosine,1,77080680,0.026653,0.973347,1.0
1,90029403,cosine,2,109930968,0.044770,0.955230,1.0
2,90029403,cosine,3,36113319,0.045816,0.954184,1.0
3,90029403,cosine,4,170119013,0.068489,0.931511,1.0
4,90029403,cosine,5,119892110,0.072546,0.927454,1.0
5,90029403,euclidean,1,77080680,0.832628,NaN,1.0
6,90029403,euclidean,2,36113319,1.096840,NaN,1.0
7,90029403,euclidean,3,109930968,1.265851,NaN,1.0
8,90029403,euclidean,4,170119013,1.396271,NaN,1.0
9,90029403,euclidean,5,51800148,1.414251,NaN,1.0


## 11. Compare clinical profiles of sampled neighbours

The table below joins the neighbour IDs to selected patient-level variables.

Use it for manual plausibility checking.

A good neighbour should generally have a coherent profile across several dimensions,
not merely match an identifier, target, or missingness pattern.

In [16]:
profile_cols = [
    c for c in [
        "patient_nbr",
        "encounter_count",
        "number_inpatient_sum",
        "number_emergency_sum",
        "number_outpatient_sum",
        "num_medications_mean",
        "number_diagnoses_mean",
        "unique_diagnosis_codes",
        "distinct_medications_recorded_active",
        "race_mode",
        "gender_mode",
        "age_mode",
        "admission_type_id_mode",
        "admission_source_id_mode",
        "A1Cresult_mode",
        "diabetesMed_mode",
    ]
    if c in train_df.columns
]

profile_lookup = train_df[profile_cols].copy()

cosine_profile = neighbor_review[
    neighbor_review["metric"] == "cosine"
][[
    "query_patient_nbr",
    "rank",
    "neighbor_patient_nbr",
    "similarity",
    "missingness_jaccard"
]].merge(
    profile_lookup,
    left_on="query_patient_nbr",
    right_on="patient_nbr",
    how="left"
).drop(columns=["patient_nbr"])

cosine_profile = cosine_profile.merge(
    profile_lookup,
    left_on="neighbor_patient_nbr",
    right_on="patient_nbr",
    how="left",
    suffixes=("_query", "_neighbor")
).drop(columns=["patient_nbr"])

display(cosine_profile)

,query_patient_nbr,rank,neighbor_patient_nbr,similarity,missingness_jaccard,encounter_count_query,number_inpatient_sum_query,number_emergency_sum_query,number_outpatient_sum_query,num_medications_mean_query,number_diagnoses_mean_query,unique_diagnosis_codes_query,distinct_medications_recorded_active_query,race_mode_query,gender_mode_query,age_mode_query,admission_type_id_mode_query,admission_source_id_mode_query,A1Cresult_mode_query,diabetesMed_mode_query,encounter_count_neighbor,number_inpatient_sum_neighbor,number_emergency_sum_neighbor,number_outpatient_sum_neighbor,num_medications_mean_neighbor,number_diagnoses_mean_neighbor,unique_diagnosis_codes_neighbor,distinct_medications_recorded_active_neighbor,race_mode_neighbor,gender_mode_neighbor,age_mode_neighbor,admission_type_id_mode_neighbor,admission_source_id_mode_neighbor,A1Cresult_mode_neighbor,diabetesMed_mode_neighbor
0,90029403,1,77080680,0.973347,1.0,1,0,0,0,10.0,9.0,3.0,1,Caucasian,Male,[70-80),2,7,>7,Yes,1,0,0,0,11.0,9.0,3.0,1,Caucasian,Male,[70-80),1,7,>7,Yes
1,90029403,2,109930968,0.955230,1.0,1,0,0,0,10.0,9.0,3.0,1,Caucasian,Male,[70-80),2,7,>7,Yes,1,0,0,0,5.0,9.0,3.0,1,Caucasian,Male,[70-80),1,7,>7,Yes
2,90029403,3,36113319,0.954184,1.0,1,0,0,0,10.0,9.0,3.0,1,Caucasian,Male,[70-80),2,7,>7,Yes,1,0,0,0,12.0,9.0,3.0,1,Caucasian,Male,[70-80),2,4,>7,Yes
3,90029403,4,170119013,0.931511,1.0,1,0,0,0,10.0,9.0,3.0,1,Caucasian,Male,[70-80),2,7,>7,Yes,1,0,0,0,9.0,9.0,3.0,1,Caucasian,Male,[70-80),1,7,>7,Yes
4,90029403,5,119892110,0.927454,1.0,1,0,0,0,10.0,9.0,3.0,1,Caucasian,Male,[70-80),2,7,>7,Yes,1,0,0,0,5.0,8.0,3.0,1,Caucasian,Male,[70-80),1,7,>7,Yes
5,41787243,1,90196155,0.879189,1.0,1,0,0,0,12.0,8.0,3.0,2,Caucasian,Female,[20-30),3,1,Norm,Yes,1,0,0,0,13.0,7.0,3.0,2,Caucasian,Female,[70-80),2,1,Norm,Yes
6,41787243,2,111346839,0.876679,1.0,1,0,0,0,12.0,8.0,3.0,2,Caucasian,Female,[20-30),3,1,Norm,Yes,1,0,0,0,13.0,9.0,2.0,2,Caucasian,Female,[70-80),3,1,Norm,Yes
7,41787243,3,16037757,0.870120,0.0,1,0,0,0,12.0,8.0,3.0,2,Caucasian,Female,[20-30),3,1,Norm,Yes,1,0,0,0,14.0,9.0,3.0,2,Caucasian,Female,[20-30),3,1,NaN,Yes
8,41787243,4,5428053,0.868111,1.0,1,0,0,0,12.0,8.0,3.0,2,Caucasian,Female,[20-30),3,1,Norm,Yes,1,0,0,0,13.0,8.0,3.0,2,Caucasian,Female,[70-80),3,6,Norm,Yes
9,41787243,5,24544206,0.862938,0.0,1,0,0,0,12.0,8.0,3.0,2,Caucasian,Female,[20-30),3,1,Norm,Yes,1,0,0,0,11.0,8.0,3.0,2,Caucasian,Female,[60-70),3,1,NaN,Yes


# 12. Select the primary similarity representation

The proposal says to test alternative definitions and reject similarity when neighbours
are dominated by artifacts.

This notebook does **not** automatically declare a winner from a single numerical score.

Instead, record:

- cosine/euclidean neighbour stability;
- missingness-pattern diagnostics;
- manual clinical plausibility.

A conservative default is **cosine similarity on the leakage-safe transformed feature
matrix**, but this should only be accepted after the manual neighbour review.

The final choice is saved as a declared configuration for Notebook 06.

In [17]:
# Conservative default configuration.
# Change ONLY after manual inspection of the saved neighbour review.
PRIMARY_SIMILARITY = "cosine"

selection_record = {
    "primary_similarity": PRIMARY_SIMILARITY,
    "feature_space": "median-imputed numeric + standardized numeric + one-hot categorical",
    "preprocessing_fit_on": "train only",
    "target_used_in_similarity": False,
    "identifier_used_in_similarity": False,
    "missingness_used_as_similarity": False,
    "dense_all_pairs_matrix_created": False,
    "manual_review_required": True,
    "manual_review_file": str(NEIGHBOR_REVIEW_PATH),
}

SELECTION_PATH = RESULTS_DIR / "05_similarity_selection.json"
SELECTION_PATH.write_text(
    json.dumps(selection_record, indent=2),
    encoding="utf-8"
)

print(json.dumps(selection_record, indent=2))

{
  "primary_similarity": "cosine",
  "feature_space": "median-imputed numeric + standardized numeric + one-hot categorical",
  "preprocessing_fit_on": "train only",
  "target_used_in_similarity": false,
  "identifier_used_in_similarity": false,
  "missingness_used_as_similarity": false,
  "dense_all_pairs_matrix_created": false,
  "manual_review_required": true,
  "manual_review_file": "c:\\Users\\Gayatri\\OneDrive\\Desktop\\sna\\results\\05_nearest_neighbor_review.csv"
}


# 13. Final checkpoint — Notebook 05

### GO criteria

- Feature matrix created successfully.
- Preprocessing fitted only on training patients.
- Validation/test are transformed using the training preprocessing.
- No patient identifier enters the feature matrix.
- No readmission/outcome variable enters the feature matrix.
- Numeric and categorical transformations are documented.
- Cosine and Euclidean neighbour searches complete.
- Similarity comparison is saved.
- Missingness-artifact audit is saved.
- Nearest-neighbour review is saved.
- No dense all-pairs similarity matrix is created.

### STOP

Stop before Notebook 06 if manual inspection finds that nearest neighbours are primarily
driven by missingness, identifiers, or other obvious artifacts.

The project proposal explicitly requires nearest-neighbour inspection and rejection of
similarity representations dominated by artifacts. fileciteturn6file2

In [18]:
FINAL_CHECK_PATH = RESULTS_DIR / "05_similarity_checkpoint.json"

final_checks = {
    "feature_matrix_train_created": X_train_sparse.shape[0] == len(train_df),
    "feature_matrix_validation_created": X_valid_sparse.shape[0] == len(valid_df),
    "feature_matrix_test_created": X_test_sparse.shape[0] == len(test_df),
    "same_feature_dimension": X_train_sparse.shape[1] == X_valid_sparse.shape[1] == X_test_sparse.shape[1],
    "preprocessing_train_only": True,
    "no_patient_id_in_features": "patient_nbr" not in candidate_feature_cols,
    "no_outcome_in_features": leakage_found == [],
    "train_matrix_finite": matrix_is_finite(X_train_sparse),
    "validation_matrix_finite": matrix_is_finite(X_valid_sparse),
    "test_matrix_finite": matrix_is_finite(X_test_sparse),
    "cosine_search_completed": cosine_indices.shape[1] == N_NEIGHBORS,
    "euclidean_search_completed": euclidean_indices.shape[1] == N_NEIGHBORS,
    "similarity_stability_saved": True,
    "missingness_audit_saved": True,
    "neighbor_review_saved": NEIGHBOR_REVIEW_PATH.exists(),
    "train_matrix_saved": TRAIN_MATRIX_PATH.exists(),
    "validation_matrix_saved": VALID_MATRIX_PATH.exists(),
    "test_matrix_saved": TEST_MATRIX_PATH.exists(),
    "feature_dictionary_saved": DICT_PATH.exists(),
    "selection_record_saved": SELECTION_PATH.exists(),
    "no_dense_all_pairs_matrix": True,
}

print("=" * 75)
print("NOTEBOOK 05 — FINAL CHECKPOINT")
print("=" * 75)

for name, passed in final_checks.items():
    print(f"{'PASS' if passed else 'FAIL':<6} | {name}")

print("=" * 75)

if all(final_checks.values()):
    print("OVERALL RESULT: PASS")
    print("Proceed to MANUAL REVIEW.")
    print("Do NOT start Notebook 06 until nearest neighbours are clinically plausible.")
else:
    print("OVERALL RESULT: FAIL")
    print("Fix the failed checks before moving to Notebook 06.")

NOTEBOOK 05 — FINAL CHECKPOINT
PASS   | feature_matrix_train_created
PASS   | feature_matrix_validation_created
PASS   | feature_matrix_test_created
PASS   | same_feature_dimension
PASS   | preprocessing_train_only
PASS   | no_patient_id_in_features
PASS   | no_outcome_in_features
PASS   | train_matrix_finite
PASS   | validation_matrix_finite
PASS   | test_matrix_finite
PASS   | cosine_search_completed
PASS   | euclidean_search_completed
PASS   | similarity_stability_saved
PASS   | missingness_audit_saved
PASS   | neighbor_review_saved
PASS   | train_matrix_saved
PASS   | validation_matrix_saved
PASS   | test_matrix_saved
PASS   | feature_dictionary_saved
PASS   | selection_record_saved
PASS   | no_dense_all_pairs_matrix
OVERALL RESULT: PASS
Proceed to MANUAL REVIEW.
Do NOT start Notebook 06 until nearest neighbours are clinically plausible.


In [19]:
summary = {
    "notebook": "05_features_similarity",
    "seed": int(SEED),
    "primary_similarity": PRIMARY_SIMILARITY,
    "candidate_feature_count": int(len(candidate_feature_cols)),
    "numeric_feature_count": int(len(numeric_cols)),
    "categorical_feature_count": int(len(categorical_cols)),
    "train_patients": int(len(train_df)),
    "validation_patients": int(len(valid_df)),
    "test_patients": int(len(test_df)),
    "transformed_feature_count": int(X_train_sparse.shape[1]),
    "cosine_neighbor_mean_jaccard_vs_euclidean": float(
        similarity_stability["mean_neighbor_jaccard"].iloc[0]
    ),
    "mean_neighbor_missingness_jaccard": float(
        missingness_audit["missingness_jaccard"].mean()
    ),
    "target_used_in_similarity": False,
    "identifier_used_in_similarity": False,
    "preprocessing_fit_on": "train only",
    "temporal_limitation_from_notebook_04": True,
    "final_checks": {str(k): bool(v) for k, v in final_checks.items()},
}

SUMMARY_PATH = RESULTS_DIR / "05_similarity_summary.json"
SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("Summary saved:", SUMMARY_PATH)

Summary saved: c:\Users\Gayatri\OneDrive\Desktop\sna\results\05_similarity_summary.json
